# FRFL — real-dataset experiment (Google Colab)

This notebook runs the repository code against your source dataset and separate A/B/C update sets. It extracts InsightFace/ArcFace embeddings, compares FedAvg against L2 drift-weighted aggregation on the same train/test split, and reports external-set metrics only when identity labels can be matched.

**Important:** use a GPU Colab runtime for faster embedding extraction. The datasets contain face images/biometric information. Keep the ZIPs in your private Google Drive; they are not uploaded to GitHub. Do not treat a random image split as identity-disjoint evaluation. Review the printed class counts and warnings before using metrics on a resume.

## Before running
Upload these ZIPs to a private folder in Google Drive, for example MyDrive/FRFL_data, and rename them exactly:
- 105_classes_pins_dataset.zip
- updated_samples_A.zip
- updated_samples_B.zip
- updated_samples_C.zip

Run every cell from top to bottom.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
DATA_DIR = Path("/content/drive/MyDrive/FRFL_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
print("Expected ZIP files in:", DATA_DIR)
print("Main dataset present:", (DATA_DIR / "105_classes_pins_dataset.zip").exists())
print("A/B/C present:", [(name, (DATA_DIR / f"updated_samples_{name}.zip").exists()) for name in "ABC"])


In [ ]:
import os, sys, json, zipfile, subprocess, shutil, time
from pathlib import Path

REPO = Path("/content/FRFL")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Tushar22-tutu/FRFL.git", str(REPO)], check=True)
os.chdir(REPO)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "insightface", "opencv-python-headless", "onnxruntime-gpu"], check=True)

import onnxruntime as ort
print("ONNX Runtime providers:", ort.get_available_providers())
print("Repo:", REPO)


In [ ]:
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
MAIN_ZIP = DATA_DIR / "105_classes_pins_dataset.zip"
assert MAIN_ZIP.exists(), "Upload/rename the main dataset ZIP in the Drive folder first."

def inspect_zip(path, preview=12):
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
    images = [n for n in names if Path(n).suffix.lower() in IMAGE_EXTS]
    print(f"{path.name}: {len(names):,} archive entries, {len(images):,} image entries")
    for name in images[:preview]:
        print("  ", name)
    return names

main_entries = inspect_zip(MAIN_ZIP)
for name in "ABC":
    p = DATA_DIR / f"updated_samples_{name}.zip"
    if p.exists():
        inspect_zip(p, preview=8)
    else:
        print("Optional archive not found:", p)


In [ ]:
RAW_DIR = Path("/content/frfl_data")
RAW_DIR.mkdir(parents=True, exist_ok=True)

def extract_zip_once(zip_path, destination):
    destination = Path(destination)
    marker = destination / ".extract_complete"
    if marker.exists():
        print("Using existing extraction:", destination)
        return destination
    destination.mkdir(parents=True, exist_ok=True)
    print("Extracting", zip_path.name, "...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(destination)
    marker.write_text("completed", encoding="utf-8")
    return destination

def find_identity_root(base, min_classes=2):
    """Find the directory whose immediate child folders contain the most images."""
    base = Path(base)
    subtree_has_images = {}
    candidates = []
    for current, dirs, files in os.walk(base, topdown=False):
        current_path = Path(current)
        has_own_images = any(Path(f).suffix.lower() in IMAGE_EXTS for f in files)
        eligible_children = [d for d in dirs if subtree_has_images.get(str(current_path / d), False)]
        has_images = has_own_images or bool(eligible_children)
        subtree_has_images[str(current_path)] = has_images
        if len(eligible_children) >= min_classes:
            candidates.append((len(eligible_children), -len(current_path.parts), current_path))
    return max(candidates, default=(0, 0, None), key=lambda item: (item[0], item[1]))[2]

def image_inventory(root, limit=30):
    root = Path(root)
    return [str(p.relative_to(root)) for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTS][:limit]

main_raw = extract_zip_once(MAIN_ZIP, RAW_DIR / "source")
SOURCE_ROOT = find_identity_root(main_raw, min_classes=2)
if SOURCE_ROOT is None:
    raise RuntimeError("Could not find identity subfolders in source ZIP. Inspect the printed archive paths and set SOURCE_ROOT manually.")
print("Detected identity root:", SOURCE_ROOT)
print("Identity folders:", sum(1 for p in SOURCE_ROOT.iterdir() if p.is_dir()))
print("Example identities:", [p.name for p in sorted(SOURCE_ROOT.iterdir()) if p.is_dir()][:12])


In [ ]:
FEATURE_DIR = RAW_DIR / "features"
subprocess.run([
    sys.executable, "scripts/extract_embeddings_insightface.py",
    "--input-dir", str(SOURCE_ROOT),
    "--output-dir", str(FEATURE_DIR),
    "--provider", "auto",
], check=True)

with open(FEATURE_DIR / "extraction_summary.json", encoding="utf-8") as f:
    source_summary = json.load(f)
print(json.dumps({
    "embedded_images": source_summary["embedded_images"],
    "identity_count": source_summary["identity_count"],
    "embedding_dimension": source_summary["embedding_dimension"],
    "providers": source_summary["providers"],
    "skipped_count": source_summary["skipped_count"],
}, indent=2))


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-v"], check=True)


In [ ]:
OUTPUT_DIR = RAW_DIR / "outputs"
subprocess.run([
    sys.executable, "train.py",
    "--x", str(FEATURE_DIR / "X.npy"),
    "--y", str(FEATURE_DIR / "y.npy"),
    "--clients", "3", "--rounds", "5", "--local-epochs", "3",
    "--mu", "0.01", "--alpha", "0.5", "--temperature", "1.0",
    "--test-size", "0.2", "--seed", "42",
    "--output", str(OUTPUT_DIR / "federated_metrics.json"),
], check=True)

report_path = OUTPUT_DIR / "federated_metrics.json"
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Dataset:", json.dumps(report["dataset"], indent=2))
for round_result in report["history"]:
    print("Round", round_result["round"])
    for method, values in round_result["methods"].items():
        print(method, values["heldout_metrics"], "weights=", [round(w, 4) for w in values["aggregation_weights"]])


In [ ]:
external_results = {}
for client in "ABC":
    archive = DATA_DIR / f"updated_samples_{client}.zip"
    if not archive.exists():
        print(f"Skipping {client}: archive not found")
        continue
    raw = extract_zip_once(archive, RAW_DIR / f"client_{client}")
    root = find_identity_root(raw, min_classes=2)
    if root is None:
        sample_paths = image_inventory(raw, limit=30)
        print(f"Skipping {client}: could not detect at least two identity folders.")
        print("Image-path examples:", sample_paths)
        print("Do not evaluate this set until each image has a verified identity label; otherwise scores would be misleading.")
        continue

    print(f"Client {client} identity root:", root)
    feature_dir = RAW_DIR / f"client_{client}_features"
    subprocess.run([
        sys.executable, "scripts/extract_embeddings_insightface.py",
        "--input-dir", str(root), "--output-dir", str(feature_dir), "--provider", "auto"
    ], check=True)
    for method in ("drift_weighted", "fedavg"):
        checkpoint = OUTPUT_DIR / f"{method}.pt"
        result_path = OUTPUT_DIR / f"external_{client}_{method}.json"
        subprocess.run([
            sys.executable, "evaluate_external.py",
            "--model", str(checkpoint), "--x", str(feature_dir / "X.npy"),
            "--y", str(feature_dir / "y.npy"), "--output", str(result_path)
        ], check=True)
        external_results[f"{client}_{method}"] = json.loads(result_path.read_text(encoding="utf-8"))

print(json.dumps(external_results, indent=2))


In [ ]:
from datetime import datetime
run_name = datetime.now().strftime("run_%Y%m%d_%H%M%S")
drive_output = DATA_DIR / "experiment_results" / run_name
drive_output.mkdir(parents=True, exist_ok=True)
for source in OUTPUT_DIR.glob("*"):
    if source.is_file():
        shutil.copy2(source, drive_output / source.name)
source_summary_path = FEATURE_DIR / "extraction_summary.json"
if source_summary_path.exists():
    shutil.copy2(source_summary_path, drive_output / source_summary_path.name)
print("Saved private experiment results to:", drive_output)
print("Upload federated_metrics.json and any external_*.json results for review before publishing the metrics.")
